In [ ]:
pip install asyncpraw


In [ ]:
import pandas as pd
import asyncpraw
from asyncpraw.models import MoreComments

In [ ]:

reddit = asyncpraw.Reddit(user_agent="Comment Extraction (by /u/doko_mitendayo)",
                     client_id="xetbErZlqeoZdm5poeleqg", client_secret="XJJVw6k7Hf-HmVwSQxpmyV2lqFVGRQ")

In [ ]:
comments_data = []

In [ ]:
# Define keywords to search for
keywords = ['LGBTQ', 'pride', 'gay', 'trans', 'lesbian', 'LGBT']

# Subreddit to search within
subreddits = ['TrueUnpopularOpinion', 'Conservative', 'Politics']

# Initialize list to store comment data
comments_data = []

# Define async function to fetch data
async def fetch_comments():
    for subreddit_name in subreddits:
        subreddit = await reddit.subreddit(subreddit_name)
        # Search posts using keywords
        async for submission in subreddit.search(' OR '.join(keywords), limit=50, syntax='lucene'):
            submission.comments.replace_more(limit=0)
            for comment in submission.comments.list():
                comments_data.append([comment.body, submission.title])

    # Create DataFrame to store data
    comments_df = pd.DataFrame(comments_data, columns=['Comment', 'Post Title'])

    # Save to CSV
    comments_df.to_csv('lgbtq_comments.csv', index=False)

import asyncio

# Check if there's an active event loop
if asyncio.get_event_loop().is_running():
    # If there's an event loop running, use create_task
    task = asyncio.create_task(fetch_comments())
else:
    # If there's no active loop, use asyncio.run
    asyncio.run(fetch_comments())

In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Download the VADER lexicon for sentiment analysis
nltk.download('vader_lexicon')

# Initialize the Sentiment Intensity Analyzer
sia = SentimentIntensityAnalyzer()

# Function to apply VADER sentiment analysis to each comment
def get_sentiment_scores(comment):
    sentiment = sia.polarity_scores(comment)
    return pd.Series([sentiment['neg'], sentiment['neu'], sentiment['pos'], sentiment['compound']])

# Apply sentiment analysis to each comment and create new columns
comments_df[['neg', 'neu', 'pos', 'compound']] = comments_df['Comment'].apply(get_sentiment_scores)

# Create a new dataframe with the sentiment columns added
all_comments = comments_df

# Save the new dataframe to a CSV file
all_comments.to_csv('lgbtq_hate_comments_with_sentiment.csv', index=False)

# Display the first few rows of the new dataframe
print(all_comments.head())

# Analyze sentiment of comments
def is_hate_speech(comment):
    sentiment = sia.polarity_scores(comment)
    return sentiment['neg'] > 0.8  # Adjust the threshold for hate detection

# Filter hateful comments
hate_comments = comments_df[comments_df['Comment'].apply(is_hate_speech)]

# Save filtered hate comments to a CSV file
hate_comments.to_csv('filtered_hate_comments3.csv', index=False)


In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Download the VADER lexicon for sentiment analysis
nltk.download('vader_lexicon')

# Initialize the Sentiment Intensity Analyzer
sia = SentimentIntensityAnalyzer()

# Analyze sentiment of comments
def is_hate_speech(comment):
    sentiment = sia.polarity_scores(comment)
    return sentiment['neg'] > 0.8  # Adjust the threshold for hate detection

# Filter hateful comments
hate_comments = comments_df[comments_df['Comment'].apply(is_hate_speech)]

# Save filtered hate comments to a CSV file
hate_comments.to_csv('filtered_hate_comments3.csv', index=False)
